# Compression-07 — Le triage par le LLM : recettes inversables d'opérations réelles

**Navigation** : [Index de la série](README.md) | [Compression-06 — MDL et complexité de Kolmogorov](Compression-06-MDL-Kolmogorov-CSharp.ipynb) | [Version Python](Compression-07-Triage-LLM-Operations-Reelles-Python.ipynb) | [07b — Triage sous Semantic Kernel](Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb)

> **Public : Recherche.** Prérequis : [Compression-03](Compression-03-LZ77-CSharp.ipynb) (fenêtre
> glissante, codecs réels) et [Compression-06](Compression-06-MDL-Kolmogorov-CSharp.ipynb) (le coût
> de la description compte autant que les données). Le carnet demande en plus l'endpoint
> OpenAI-compatible de la pile vLLM locale, décrit dans sa cinquième partie.

Ce carnet est le jumeau C# du [07 Python](Compression-07-Triage-LLM-Operations-Reelles-Python.ipynb) :
même séquence de cellules, même banc, mêmes contrats. Il est l'accrétion réelle du pli **05b**
(`Compression-05b-NeuroSymbolic-Recipe`). Le 05b démontrait le principe sur un DSL jouet : une
recette écrite dans un petit langage, exécutée par un interpréteur borné, certifiée par une
empreinte. Ici, le DSL devient une **bibliothèque d'opérations réelles** appliquées à des
**fichiers** et à leurs **segments**, et le triage est confié au **LLM local** — le
**Swift-1.5-Qwen3.8-27B (W4A16-AWQ)** servi par la pile vLLM de la flotte —, en *function
calling*. Côté codecs, le jumeau n'appelle pas Python : `zlib` vient de la BCL .NET
(`ZLibStream`), `bz2` et `lzma` de deux bibliothèques .NET épinglées (SharpZipLib, LZMA-SDK) —
les tailles compressées diffèrent donc légèrement du Python, et les verdicts se lisent sur les
mêmes bancs, chiffres du carnet à l'appui.

Ce que le carnet démontre, dans l'ordre :

1. un vocabulaire d'opérations **réellement inversables** — quatre codecs réels (`zlib`, `bz2`,
   `lzma`, `store`) et quatre découpages qui qualifient les sous-segments ;
2. un **harnais** où le LLM voit le découpage des fichiers (taille, entropie, aperçu) et choisit
   une opération par segment, par appel d'outil ;
3. une **boucle de rappel** : chaque segment produit par un découpage revient devant le LLM
   jusqu'à ce qu'il ait reçu un traitement final ;
4. une **décompression sans LLM** : la recette certifiée est rejouée à l'envers, et le banc
   vérifie le round-trip octet pour octet ;
5. un **banc comparatif** : le triage contre le traitement indifférencié d'un bloc, puis contre
   le meilleur codec global choisi après coup — une borne favorable, puisque personne ne choisit
   le meilleur codec sans les essayer tous.

**Prérequis d'exécution** : l'endpoint OpenAI-compatible de la pile vLLM locale. Il sert le
**Swift-1.5-Qwen3.8-27B (W4A16-AWQ)** sous les alias d'endpoint `swift-1.5-27b` et
`qwen3.6-35b-a3b` : le champ `model` d'une réponse renvoie le **premier nom servi**
(`qwen3.6-35b-a3b`), jamais le checkpoint chargé — demander `swift-1.5-27b` fait répondre la
pile sous `qwen3.6-35b-a3b` (le [07b](Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb)
mesure ce piège en détail). La clé d'API vient de la variable d'environnement
`VLLM_API_KEY_MEDIUM`, jamais du carnet. Le carnet échoue explicitement si l'endpoint est
absent — il n'existe pas de sortie de repli : une trace de triage sans LLM ne prouverait rien.

## Ce que le carnet mesure, et ce qu'il ne mesure pas

Le banc de la dernière partie compare trois traitements du **même** contenu :

- le traitement indifférencié d'un bloc : tout concaténé, un seul codec (c'est ce que fait un
  `tar.gz` quand personne ne choisit) ;
- le même bloc avec le **meilleur** codec global, choisi après coup (une borne favorable : on
  offre au traitement naïf un oracle qu'il n'a pas) ;
- le **triage par segment** : le LLM choisit un codec par fichier et découpe ce qui mérite de
  l'être, chaque sous-segment recevant son propre codec.

Le coût de la recette (les décisions elles-mêmes, sérialisées) est **compté** dans le total du
triage : une recette qui ne se paierait pas serait une comptabilité fausse. Ce que le carnet ne
mesure pas : le temps de décision du LLM, qui n'est pas de la compression mais du triage, et qui
est rapporté à part.

In [1]:
#r "nuget: SharpZipLib, 1.4.2"
#r "nuget: LZMA-SDK, 22.1.1"

using System;
using System.Diagnostics;
using System.Globalization;
using System.IO;
using System.IO.Compression;
using System.Linq;
using System.Net.Http;
using System.Net.Http.Headers;
using System.Security.Cryptography;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using ICSharpCode.SharpZipLib.BZip2;
using SevenZip.Compression.LZMA;

// Canon de la serie : l'endpoint vLLM de la pile locale, la cle hors du carnet.
const string BaseUrl = "http://192.168.0.47:5002/v1";

string[] MODELES_ATTENDUS = { "swift-1.5-27b", "qwen3.6-35b-a3b" };

static string ApiKey()
{
    var cle = Environment.GetEnvironmentVariable("VLLM_API_KEY_MEDIUM");
    if (string.IsNullOrEmpty(cle))
        throw new InvalidOperationException(
            "VLLM_API_KEY_MEDIUM absente de l'environnement : jamais de cle dans le carnet.");
    return cle;
}

static readonly HttpClient Http = new HttpClient { Timeout = TimeSpan.FromSeconds(180) };

static List<string> ModelesDisponibles()
{
    using var requete = new HttpRequestMessage(HttpMethod.Get, BaseUrl + "/models");
    requete.Headers.Authorization = new AuthenticationHeaderValue("Bearer", ApiKey());
    using var reponse = Http.Send(requete);
    reponse.EnsureSuccessStatusCode();
    using var doc = JsonDocument.Parse(reponse.Content.ReadAsStream());
    return doc.RootElement.GetProperty("data")
        .EnumerateArray()
        .Select(m => m.GetProperty("id").GetString())
        .ToList();
}

List<string> MODELES = ModelesDisponibles();
string MODELE = MODELES.FirstOrDefault(m => m.Contains("swift"))
    ?? throw new InvalidOperationException(
        $"aucun modele swift sur {BaseUrl} ; modeles vus : [{string.Join(", ", MODELES)}]");
Console.WriteLine($"endpoint {BaseUrl}");
Console.WriteLine($"modeles  [{string.Join(", ", MODELES)}]");
Console.WriteLine($"modele de triage : {MODELE}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages LZMA-SDK, 22.1.1 SharpZipLib, 1.4.2

endpoint http://192.168.0.47:5002/v1


modeles  [qwen3.6-35b-a3b, swift-1.5-27b]


modele de triage : swift-1.5-27b


## 1. Le vocabulaire : segments et opérations

Un **segment** est un bloc d'octets nommé, avec un type et un état. Un fichier du répertoire est
un segment racine ; un découpage en produit des enfants, qui héritent du nom de leur parent.

Le LLM ne voit pas les octets : il voit une **vue** — identifiant, taille, type, entropie
(shannonienne, en bits par octet) et un aperçu imprimable. C'est cette vue qui porte la décision :
un contenu à 7,9 bits par octet ne se comprime pas, un journal à 4,8 bits se comprime.

In [2]:
public class Segment
{
    /// Un bloc d'octets nomme, avec un type et un etat de traitement.
    public string Id;
    public byte[] Data;
    public string Kind;
    public string Parent;
    public string State = "pending";          // pending -> terminal

    public Segment(string id, byte[] data, string kind = "octets", string parent = null)
    {
        Id = id;
        Data = data;
        Kind = kind;
        Parent = parent;
    }

    public override string ToString() => $"<{Id} {Kind} {Data.Length}o {State}>";
}

static double Entropie(byte[] data)
{
    /// Entropie de Shannon en bits par octet : la borne basse d'un codec sans modele.
    if (data.Length == 0) return 0.0;
    return -data.GroupBy(b => b)
        .Sum(g => { double p = (double)g.Count() / data.Length; return p * Math.Log2(p); });
}

static string Apercu(byte[] data, int n = 110)
{
    return new string(data.Take(n).Select(b => b >= 32 && b < 127 ? (char)b : '.').ToArray());
}

static Dictionary<string, object> Vue(Segment seg)
{
    /// Ce que le LLM voit d'un segment : jamais les octets, seulement de quoi decider.
    var info = new Dictionary<string, object>
    {
        ["id"] = seg.Id,
        ["octets"] = seg.Data.Length,
        ["type"] = seg.Kind,
        ["entropie_bits_par_octet"] = Math.Round(Entropie(seg.Data), 2),
    };
    if (seg.Data.Take(4096).Contains((byte)'\n'))
        info["lignes"] = seg.Data.Count(b => b == (byte)'\n');
    info["apercu"] = Apercu(seg.Data);
    return info;
}

Console.WriteLine(new Segment("exemple", Encoding.UTF8.GetBytes("vue d'un segment"), "demonstration"));

<exemple demonstration 16o pending>


## 2. Les primitives : quatre codecs réels

Aucune opération n'est simulée : `zlib` vient de la BCL (`ZLibStream`, niveau maximal), `bz2` de
SharpZipLib et `lzma` du SDK 7-Zip — les mêmes familles de codecs que celles des carnets 01 à 04b
de la série, qui les construisent ou les comparent. `store` est l'identité : elle est une
opération à part entière, parce que recompresser un contenu déjà compressé le fait **grossir**.

Une primitive transforme un segment en un segment compressé binaire. Elle est inversable par
construction : `DECODEURS` est la table inverse, et c'est elle — jamais le LLM — qui servira à
décompresser.

In [3]:
static byte[] ZlibCompresse(byte[] d)
{
    using var ms = new MemoryStream();
    using (var z = new ZLibStream(ms, CompressionLevel.SmallestSize, true)) z.Write(d, 0, d.Length);
    return ms.ToArray();
}

static byte[] ZlibDecomprime(byte[] b)
{
    using var ms = new MemoryStream(b);
    using var z = new ZLibStream(ms, CompressionMode.Decompress);
    using var sortie = new MemoryStream();
    z.CopyTo(sortie);
    return sortie.ToArray();
}

static byte[] Bz2Compresse(byte[] d)
{
    using var ms = new MemoryStream();
    using (var bz = new BZip2OutputStream(ms, 9)) bz.Write(d, 0, d.Length);
    return ms.ToArray();
}

static byte[] Bz2Decomprime(byte[] b)
{
    using var ms = new MemoryStream(b);
    using var bz = new BZip2InputStream(ms);
    using var sortie = new MemoryStream();
    bz.CopyTo(sortie);
    return sortie.ToArray();
}

// lzma : cadre "seul" a la 7-Zip — 5 octets de proprietes + 8 octets de taille + flux.
static byte[] LzmaCompresse(byte[] d)
{
    var enc = new SevenZip.Compression.LZMA.Encoder();
    using var sortie = new MemoryStream();
    enc.WriteCoderProperties(sortie);
    sortie.Write(BitConverter.GetBytes((long)d.Length), 0, 8);
    using var entree = new MemoryStream(d);
    enc.Code(entree, sortie, d.LongLength, -1L, null);
    return sortie.ToArray();
}

static byte[] LzmaDecomprime(byte[] b)
{
    var props = new byte[5];
    Array.Copy(b, 0, props, 0, 5);
    long taille = BitConverter.ToInt64(b, 5);
    var dec = new SevenZip.Compression.LZMA.Decoder();
    dec.SetDecoderProperties(props);
    using var entree = new MemoryStream(b, 13, b.Length - 13);
    using var sortie = new MemoryStream();
    dec.Code(entree, sortie, -1L, taille, null);
    return sortie.ToArray();
}

delegate byte[] Codec(byte[] b);

Dictionary<string, Codec> CODECS = new()
{
    ["zlib"] = ZlibCompresse,
    ["bz2"] = Bz2Compresse,
    ["lzma"] = LzmaCompresse,
    ["store"] = b => b,
};
Dictionary<string, Codec> DECODEURS = new()
{
    ["zlib"] = ZlibDecomprime,
    ["bz2"] = Bz2Decomprime,
    ["lzma"] = LzmaDecomprime,
    ["store"] = b => b,
};

// la meme charge, vue par les quatre codecs : le choix n'est pas neutre
var motif = Encoding.UTF8.GetBytes("ligne de journal standard, repetee avec un compteur ");
var bruit = RandomNumberGenerator.GetBytes(256);
byte[] ECHANTILLON = Enumerable.Repeat(motif, 40).SelectMany(x => x).Concat(bruit).ToArray();
foreach (var (nom, codec) in CODECS)
    Console.WriteLine($"{nom,-5} {ECHANTILLON.Length,6} o -> {codec(ECHANTILLON).Length,6} o");

zlib    2336 o ->    348 o


bz2     2336 o ->    530 o


lzma    2336 o ->    349 o


store   2336 o ->   2336 o


## 3. Les opérations intermédiaires : découper pour qualifier

Une opération intermédiaire ne compresse rien : elle **produit des segments qualifiés** à partir
d'un segment macro, et chacun repart dans la boucle de traitement. Quatre découpages, tous
inversables :

| stratégie | ce qu'elle produit | pourquoi elle peut gagner |
|---|---|---|
| `lines` | une ligne par segment | séparer un en-tête rare d'un corps répétitif |
| `blocks` | des blocs de 4 Kio | borner la mémoire d'un très gros fichier |
| `csv_columns` | l'en-tête puis une colonne par segment | une colonne numérique ou catégorielle se comprime bien mieux seule |
| `json_keys` | une clé de premier niveau par segment | des clés de natures différentes, des codecs différents |

L'inverse est porté par les **identifiants** des enfants : `f1.col2` sait qu'il est la colonne 2
de `f1`, `f1.k_mesures` sait qu'il est la clé `mesures`. La recette n'a donc pas à recopier la
structure — elle nomme les enfants, et le rejeu la reconstruit.

Un découpage n'est pas toujours applicable, et le harnais le **vérifie** au lieu de le supposer :
un `csv_columns` sur un fichier sans virgule, un `json_keys` sur une racine qui est une liste, un
`lines` qui produirait des centaines de segments échouent et rendent une raison. C'est cette raison
que le modèle recevra pour se corriger.

In [4]:
string[] STRATEGIES = { "lines", "blocks", "csv_columns", "json_keys" };
const int MAX_ENFANTS = 12;   // un decoupage qui pulverise un fichier coute un appel par morceau

// le rejeu reserialise le JSON avec la meme mise en forme que la source du banc (indent 1) :
// un JSON formate autrement se stocke tel quel plutot que d'etre decoupe
static readonly JsonSerializerOptions JSON_INDENT1 = new() { WriteIndented = true, IndentSize = 1 };

static List<byte[]> CouperLignes(byte[] d)
{
    var res = new List<byte[]>();
    int debut = 0;
    for (int i = 0; i < d.Length; i++)
        if (d[i] == (byte)'\n') { res.Add(d[debut..(i + 1)]); debut = i + 1; }
    if (debut < d.Length) res.Add(d[debut..]);
    return res;
}

static List<Segment> Decouper(Segment seg, string strategie)
{
    /// Operation intermediaire : un segment macro -> des segments qualifies.
    /// Leve InvalidOperationException si l'operation ne s'applique pas : le harnais refuse une
    /// operation qui ne serait pas inversable sur ce contenu, et le dit au modele.
    var enfants = new List<Segment>();
    if (strategie == "lines")
    {
        var lignes = CouperLignes(seg.Data);
        if (lignes.Count > MAX_ENFANTS)
            throw new InvalidOperationException($"{lignes.Count} lignes : au-dela de {MAX_ENFANTS} segments, " +
                "compresser le fichier entier coute moins d'appels");
        for (int i = 0; i < lignes.Count; i++)
            enfants.Add(new Segment($"{seg.Id}.{i}", lignes[i], "ligne", seg.Id));
    }
    else if (strategie == "blocks")
    {
        for (int i = 0; i < Math.Max(seg.Data.Length, 1); i += 4096)
            enfants.Add(new Segment($"{seg.Id}.{i / 4096}", seg.Data[i..Math.Min(i + 4096, seg.Data.Length)], "bloc", seg.Id));
        if (enfants.Count > MAX_ENFANTS)
            throw new InvalidOperationException($"{enfants.Count} blocs : au-dela de {MAX_ENFANTS} segments, " +
                "compresser le fichier entier coute moins d'appels");
    }
    else if (strategie == "csv_columns")
    {
        string texte = new UTF8Encoding(false, true).GetString(seg.Data);   // strict : un binaire echoue ici
        var lignes = texte.Split('\n').Where(l => l.Trim().Length > 0).ToList();
        if (lignes.Count < 2)
            throw new InvalidOperationException("moins de deux lignes : il n'y a pas de tableau a decouper");
        var entetes = lignes[0].Split(',').ToList();
        if (entetes.Count < 2)
            throw new InvalidOperationException("en-tete sans virgule : ce fichier n'est pas un CSV");
        foreach (var ligne in lignes.Skip(1))
            if (ligne.Split(',').Length != entetes.Count)
                throw new InvalidOperationException("lignes de largeurs inegales : le decoupage ne serait pas inversable");
        enfants.Add(new Segment($"{seg.Id}.hdr", Encoding.UTF8.GetBytes(lignes[0]), "entete-csv", seg.Id));
        var colonnes = Enumerable.Range(0, entetes.Count).Select(_ => new List<string>()).ToList();
        foreach (var ligne in lignes.Skip(1))
        {
            var champs = ligne.Split(',');
            for (int j = 0; j < colonnes.Count; j++) colonnes[j].Add(champs[j]);
        }
        for (int j = 0; j < entetes.Count; j++)
            enfants.Add(new Segment($"{seg.Id}.col{j}", Encoding.UTF8.GetBytes(string.Join("\n", colonnes[j])),
                $"colonne:{entetes[j].Trim()}", seg.Id));
    }
    else if (strategie == "json_keys")
    {
        var charge = JsonNode.Parse(new UTF8Encoding(false, true).GetString(seg.Data));
        if (charge is not JsonObject)
            throw new InvalidOperationException($"racine {charge.GetType().Name} : json_keys attend un objet a cles");
        foreach (var (cle, valeur) in (JsonObject)charge)
            enfants.Add(new Segment($"{seg.Id}.k_{cle}", Encoding.UTF8.GetBytes(valeur.ToJsonString()), $"cle:{cle}", seg.Id));
    }
    else
        throw new InvalidOperationException($"strategie inconnue : {strategie}");
    return enfants;
}

static byte[] ReconstruireEnfants(Dictionary<string, object> op, List<byte[]> donnees)
{
    /// Rejoue l'inverse d'un decoupage, sans LLM et sans decision.
    var strategie = (string)op["strategie"];
    var ids = (List<string>)op["enfants"];
    if (strategie == "lines" || strategie == "blocks")
        return donnees.SelectMany(d => d).ToArray();
    if (strategie == "csv_columns")
    {
        string entete = Encoding.UTF8.GetString(donnees[0]);
        var colonnes = donnees.Skip(1).Select(d => Encoding.UTF8.GetString(d).Split('\n').ToList()).ToList();
        int n = colonnes.Max(c => c.Count);
        var lignes = Enumerable.Range(0, n).Select(i => string.Join(",", colonnes.Select(c => i < c.Count ? c[i] : "")));
        return Encoding.UTF8.GetBytes(string.Join("\n", new[] { entete }.Concat(lignes)) + "\n");
    }
    if (strategie == "json_keys")
    {
        var obj = new JsonObject();
        for (int i = 0; i < ids.Count; i++)
        {
            // dernier ".k_" : un decoupage de cle peut lui-meme etre decoupe
            string sid = ids[i];
            obj[sid[(sid.LastIndexOf(".k_") + 3)..]] = JsonNode.Parse(Encoding.UTF8.GetString(donnees[i]));
        }
        return Encoding.UTF8.GetBytes(JsonSerializer.Serialize(obj, JSON_INDENT1));
    }
    throw new InvalidOperationException(strategie);
}

## 4. Le banc : un répertoire hétérogène

Un répertoire qui mélange les natures, comme un vrai répertoire de travail :

- un **journal** à lignes répétitives, faible entropie ;
- un **CSV** à colonnes de natures très différentes — un identifiant, une catégorie à cinq
  valeurs, un montant, un commentaire ;
- un **JSON** à clés de natures différentes lui aussi ;
- un **binaire déjà compressé** — le piège classique : le recompresser le fait grossir.

Les contenus sont générés par le carnet, avec une graine fixe : le banc est rejouable.

In [5]:
// Corpus de test rejouable : quatre fichiers de natures differentes (journal redondant,
// CSV a colonnes heterogenes, JSON structure, binaire deja compresse), generes de facon
// deterministe pour que le banc reste comparable d'une execution a l'autre.
static Dictionary<string, byte[]> CorpusTest()
{
    var fichiers = new Dictionary<string, byte[]>();
    fichiers["journal.log"] = Encoding.UTF8.GetBytes(
        string.Join("\n", Enumerable.Range(0, 800).Select(i =>
            $"2026-11-14T12:{i % 60:00}:00Z INFO worker {i % 4} tick {i} status=ok latency={i % 7}ms")) + "\n");
    string entetes = "id,categorie,montant,commentaire";
    var lignes = Enumerable.Range(0, 1200).Select(i =>
        $"{i},client_{i % 5},{(((i * 37) % 1000) / 10.0).ToString("F1", CultureInfo.InvariantCulture)}," +
        $"ligne de commentaire standard numero {i % 11}");
    fichiers["ventes.csv"] = Encoding.UTF8.GetBytes(entetes + "\n" + string.Join("\n", lignes) + "\n");
    var obj = new JsonObject
    {
        ["entete"] = new JsonObject { ["version"] = 3, ["producteur"] = "banc-compression-07" },
        ["mesures"] = new JsonArray(Enumerable.Range(0, 400).Select(i => (JsonNode)new JsonObject
        {
            ["capteur"] = $"c{i % 6}",
            ["valeur"] = Math.Round(((i * 13) % 100) / 3.0, 2),
        }).ToArray()),
        ["journal"] = new JsonArray(Enumerable.Range(0, 120).Select(_ => (JsonNode)"evenement standard").ToArray()),
    };
    fichiers["mesures.json"] = Encoding.UTF8.GetBytes(JsonSerializer.Serialize(obj, JSON_INDENT1));
    fichiers["vrac.bin"] = ZlibCompresse(RandomNumberGenerator.GetBytes(6144));
    return fichiers;
}

Dictionary<string, byte[]> FICHIERS = CorpusTest();
int total = FICHIERS.Values.Sum(v => v.Length);
foreach (var (nom, donnees) in FICHIERS)
    Console.WriteLine($"{nom,-14} {donnees.Length,7} o  entropie {Entropie(donnees).ToString("F2", CultureInfo.InvariantCulture)} bits/octet");
Console.WriteLine($"{"total",-14} {total,7} o");

journal.log      52690 o  entropie 4.82 bits/octet


ventes.csv       68515 o  entropie 4.51 bits/octet


mesures.json     23388 o  entropie 4.16 bits/octet


vrac.bin          6155 o  entropie 7.97 bits/octet


total           150748 o


## 5. Le harnais : function calling sur le Swift local

Le LLM n'écrit pas de code : il **appelle deux fonctions** déclarées en JSON Schema, exactement
comme dans `GenAI/Texte/04_Function_Calling.ipynb` et `18_Semantic_Kernel_Plugins.ipynb`, mais
ici par l'API `tools` d'un endpoint OpenAI-compatible servi localement.

Deux détails mesurés sur ce modèle, et qui comptent :

- le modèle **raisonne** avant d'appeler l'outil ; il faut lui laisser un budget de jetons
  suffisant, sinon la réponse revient vide (le raisonnement a consommé le budget) ;
- il arrive qu'il recopie la vue entière dans le champ `segment` au lieu de l'identifiant : le
  harnais normalise en relisant l'identifiant qu'il a recopié, et **compte** ces normalisations,
  parce que c'est un défaut du harnais, pas du triage.

In [6]:
// Harnais LLM : les deux outils declares en JSON Schema (compresser un segment, decouper
// un segment selon sa structure), le prompt systeme qui arbitre entre eux, et le client
// HTTP qui poste chaque appel en mesurant sa latence et son usage de jetons.
const string OUTILS_JSON = """
[
  {
    "type": "function",
    "function": {
      "name": "compresser_segment",
      "description": "Compresse definitivement un segment avec un codec reversible.",
      "parameters": {
        "type": "object",
        "properties": {
          "segment": {"type": "string", "description": "identifiant exact du segment"},
          "codec": {"type": "string", "enum": ["zlib", "bz2", "lzma", "store"]}
        },
        "required": ["segment", "codec"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "decouper_segment",
      "description": "Decoupe un segment macro en segments qualifies, traites ensuite un par un. csv_columns : l'en-tete puis une colonne par segment (tableau a colonnes de natures differentes). json_keys : une cle de premier niveau par segment. lines : une ligne par segment. blocks : des blocs de 4 Kio.",
      "parameters": {
        "type": "object",
        "properties": {
          "segment": {"type": "string", "description": "identifiant exact du segment"},
          "strategie": {"type": "string", "enum": ["lines", "blocks", "csv_columns", "json_keys"]}
        },
        "required": ["segment", "strategie"]
      }
    }
  }
]
""";

const string SYSTEME = """
Tu es un trieur de compression. Pour chaque segment, choisis UNE operation reversible, en lisant son apercu et son entropie.
- L'apercu montre un tableau CSV (une ligne d'en-tete, puis des lignes separees par des virgules) : decoupe-le avec csv_columns. Ses colonnes ont des natures tres differentes (identifiants, categories, montants, texte libre) et se compressent bien mieux separement que entremeslees.
- L'apercu montre un objet JSON (des cles suivies de valeurs) : decoupe-le avec json_keys.
- L'apercu ne montre pas de structure (texte courant, binaire) : compresse le segment entier.
Pour un codec : lzma est en general le plus econome sur du texte redondant, bz2 vient ensuite, zlib est le plus faible ; store est le choix d'un contenu deja compresse ou aleatoire, que recompresser ferait grossir.
Chaque segment produit coute un appel de plus : ne decoupe que ce dont la structure est visible, et jamais ligne a ligne un fichier de plusieurs dizaines de Ko.
Reponds par un appel d'outil, jamais par du texte.
""";

const int JETONS_MAX = 900;
List<JsonNode> APPELS = new();

(JsonObject message, double latence) AppelLlm(JsonArray messages)
{
    var payload = new JsonObject
    {
        ["model"] = MODELE,
        ["messages"] = messages.DeepClone(),
        ["tools"] = JsonNode.Parse(OUTILS_JSON),
        ["tool_choice"] = "required",
        ["temperature"] = 0,
        ["max_tokens"] = JETONS_MAX,
    };
    using var requete = new HttpRequestMessage(HttpMethod.Post, BaseUrl + "/chat/completions")
    {
        Content = new StringContent(payload.ToJsonString(), Encoding.UTF8, "application/json"),
    };
    requete.Headers.Authorization = new AuthenticationHeaderValue("Bearer", ApiKey());
    var chrono = Stopwatch.StartNew();
    using var reponse = Http.Send(requete);
    reponse.EnsureSuccessStatusCode();
    var charge = JsonNode.Parse(reponse.Content.ReadAsStream());
    APPELS.Add(charge["usage"].DeepClone());
    chrono.Stop();
    return ((JsonObject)charge["choices"][0]["message"], chrono.Elapsed.TotalSeconds);
}

// un appel de controle, sur un segment fabrique : le harnais parle bien au modele
var temoin = new Segment("temoin", Encoding.UTF8.GetBytes(string.Concat(Enumerable.Repeat("aaaa bbbb aaaa bbbb ", 64))), "echantillon");
var (messageTemoin, latenceTemoin) = AppelLlm(new JsonArray
{
    new JsonObject { ["role"] = "system", ["content"] = SYSTEME },
    new JsonObject { ["role"] = "user", ["content"] = "Segment a traiter : " + JsonSerializer.Serialize(Vue(temoin)) },
});
Console.WriteLine($"appel temoin en {latenceTemoin.ToString("F1", CultureInfo.InvariantCulture)}s : {messageTemoin["tool_calls"]?.ToJsonString()}");

appel temoin en 7.8s : [{"id":"chatcmpl-tool-95f15ed38f15652d","type":"function","function":{"name":"compresser_segment","arguments":"{\u0022segment\u0022: \u0022temoin\u0022, \u0022codec\u0022: \u0022lzma\u0022}"}}]


## 6. La boucle de rappel

C'est le cœur du contrat : **le LLM est rappelé jusqu'à ce que tous les segments aient reçu un
traitement final**. Un découpage ne termine rien — il crée du travail ; les enfants repartent donc
en file, et la boucle ne s'arrête que lorsque la file est vide.

Le rappel sert aussi à **réparer** : quand le modèle demande une opération que le contenu refuse
(un `json_keys` sur une liste, un `csv_columns` sans virgule), le harnais ne l'exécute pas, il
renvoie la raison au modèle et lui redemande une décision. Ces réparations sont comptées — c'est
un défaut du triage, pas du harnais, et un banc qui les tairait surestimerait le modèle.

Un budget d'appels borne la boucle : un triage qui tourne indéfiniment serait une panne, pas une
découverte. Il n'y a pas de repli silencieux — un segment que le budget n'atteint pas fait échouer
la cellule, avec son nom.

In [7]:
List<(string seg, string op, string raison)> REPARATIONS = new();  // le modele s'est corrige
List<string> NORMALISATIONS = new();   // segments ou le modele a recopie la vue au lieu de l'identifiant
const int MAX_APPELS = 30;

(string nom, JsonObject args, double latence) Decider(Segment seg, string note = "")
{
    /// Un appel au modele : la vue du segment, et la raison d'un refus precedent s'il y en a une.
    var contenu = $"Segment a traiter (identifiant exact : \"{seg.Id}\").\n"
        + JsonSerializer.Serialize(Vue(seg))
        + $"\nLe champ \"segment\" doit valoir exactement \"{seg.Id}\".";
    if (note.Length > 0) contenu += "\n" + note;
    var (message, latence) = AppelLlm(new JsonArray
    {
        new JsonObject { ["role"] = "system", ["content"] = SYSTEME },
        new JsonObject { ["role"] = "user", ["content"] = contenu },
    });
    var appels = message["tool_calls"] as JsonArray;
    if (appels is null || appels.Count == 0)
        throw new InvalidOperationException($"le modele n'a pas appele d'outil sur {seg.Id} : {message["content"]}");
    var fonction = (JsonObject)appels[0]["function"];
    var args = (JsonObject)JsonNode.Parse((string)fonction["arguments"]);
    if ((string?)args["segment"] != seg.Id)
    {
        // le modele recopie parfois la vue entiere : on relit l'identifiant qu'il a recopie
        var brut = (string?)args["segment"] ?? "";
        try { args["segment"] = JsonNode.Parse(brut)?["id"] ?? seg.Id; }
        catch (JsonException) { args["segment"] = seg.Id; }
        NORMALISATIONS.Add(seg.Id);
    }
    return ((string)fonction["name"], args, latence);
}

(Dictionary<string, object> op, List<Segment> enfants) Appliquer(Segment seg, string nom, JsonObject args)
{
    /// Execute la decision du modele ; leve si l'operation ne s'applique pas au contenu.
    if (nom == "compresser_segment")
    {
        var codec = (string?)args["codec"] ?? "zlib";
        seg.State = "terminal";
        return (new Dictionary<string, object> { ["op"] = "compress", ["seg"] = seg.Id, ["codec"] = codec },
                new List<Segment>());
    }
    var strategie = (string?)args["strategie"] ?? "lines";
    var enfants = Decouper(seg, strategie);
    return (new Dictionary<string, object>
    {
        ["op"] = "split", ["seg"] = seg.Id, ["strategie"] = strategie,
        ["enfants"] = enfants.Select(e => e.Id).ToList(),
    }, enfants);
}

(List<Dictionary<string, object>> ops, Dictionary<string, Segment> index) Trier(
    List<Segment> racines, int maxAppels = MAX_APPELS, bool verbeux = true)
{
    /// Boucle de rappel : chaque segment pending recoit un traitement final, par le LLM.
    var ops = new List<Dictionary<string, object>>();
    var index = racines.ToDictionary(s => s.Id);
    var pending = new List<Segment>(racines);
    while (pending.Count > 0 && APPELS.Count < maxAppels)
    {
        var seg = pending[0];
        pending.RemoveAt(0);
        var (nom, args, latence) = Decider(seg);
        Dictionary<string, object> op;
        List<Segment> enfants;
        try
        {
            (op, enfants) = Appliquer(seg, nom, args);
        }
        catch (Exception err) when (err is InvalidOperationException or JsonException or DecoderFallbackException)
        {
            REPARATIONS.Add((seg.Id, nom, err.Message));
            if (verbeux)
                Console.WriteLine($"  {seg.Id,-14} refus : {nom} -- {err.Message}");
            var note = $"L'operation {nom} a echoue sur ce segment : {err.Message}. " +
                "Choisis une operation qui s'applique reellement a ce contenu.";
            (nom, args, latence) = Decider(seg, note);
            (op, enfants) = Appliquer(seg, nom, args);      // un second echec est fatal, et c'est voulu
        }
        ops.Add(op);
        foreach (var enfant in enfants) index[enfant.Id] = enfant;
        pending.AddRange(enfants);
        if (verbeux)
        {
            if ((string)op["op"] == "compress")
            {
                int taille = CODECS[(string)op["codec"]](seg.Data).Length;
                Console.WriteLine($"  {seg.Id,-14} {seg.Data.Length,7} o -> {(string)op["codec"],-5} {taille,7} o  ({latence.ToString("F1", CultureInfo.InvariantCulture)}s)");
            }
            else
            {
                Console.WriteLine($"  {seg.Id,-14} {seg.Data.Length,7} o -> {(string)op["strategie"]} " +
                    $"({enfants.Count} enfants, {latence.ToString("F1", CultureInfo.InvariantCulture)}s)");
            }
        }
    }
    if (pending.Count > 0)
        throw new InvalidOperationException($"budget de {maxAppels} appels epuise : " +
            $"{pending.Count} segments sans traitement final (dont {pending[0].Id})");
    if (NORMALISATIONS.Count > 0)
        Console.WriteLine($"  identifiants recopies puis normalises : {NORMALISATIONS.Count}");
    if (REPARATIONS.Count > 0)
        Console.WriteLine($"  operations refusees puis corrigees : {REPARATIONS.Count}");
    return (ops, index);
}


(22,16): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(25,27): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(38,28): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(43,28): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## 7. Triage du banc

Le LLM voit les quatre fichiers du banc, un par un. Ce qu'il décide est affiché à mesure — et
c'est cette trace qui rend le triage vérifiable : une décision qu'on ne peut pas lire n'est pas
une décision, c'est un effet.

In [8]:
var racines = FICHIERS.Select((kv, i) => new Segment($"f{i}", kv.Value, $"fichier:{kv.Key}")).ToList();
var chronoTriage = Stopwatch.StartNew();
var (OPS, INDEX) = Trier(racines);
chronoTriage.Stop();
double DUREE_TRIAGE = chronoTriage.Elapsed.TotalSeconds;
Console.WriteLine($"\n{OPS.Count} operations, {APPELS.Count} appels au modele, {DUREE_TRIAGE.ToString("F1", CultureInfo.InvariantCulture)}s");

  f0               52690 o -> lzma     1632 o  (12.6s)


  f1               68515 o -> csv_columns (5 enfants, 11.9s)


  f2               23388 o -> json_keys (3 enfants, 6.9s)


  f3                6155 o -> store    6155 o  (7.9s)


  f1.hdr         refus : decouper_segment -- moins de deux lignes : il n'y a pas de tableau a decouper


  f1.hdr              32 o -> store      32 o  (12.6s)


... (8 outputs left out)

  f2.k_entete.k_version       1 o -> store       1 o  (8.5s)


  f2.k_entete.k_producteur      21 o -> lzma       39 o  (7.4s)


  operations refusees puis corrigees : 2



14 operations, 17 appels au modele, 181.9s


### Lecture.

La trace ci-dessus est le triage lui-même. Elle se lit ligne à ligne, et ce qu'on y cherche n'est
pas un résultat mais une **décision** : quel segment a été compressé tel quel, lequel a été
découpé et en quoi. Le binaire déjà compressé doit finir en `store` — s'il reçoit un codec, la
mesure de la dernière partie le sanctionnera, puisque le total inclura son alourdissement.

Deux événements de la trace méritent l'arrêt. D'abord les **refus** : une opération qui ne
s'applique pas au contenu — un découpage de lignes sur un en-tête d'une seule ligne, un
`json_keys` sur une racine qui est une liste — est refusée par le harnais avec sa raison, et
cette raison repart vers le modèle qui choisit autre chose. Un refus suivi d'une reprise est le
signe que la boucle fonctionne : le modèle a vu son erreur, il ne l'a pas reproduite. Ensuite les
**découpages imbriqués** : un segment produit par un découpage peut lui-même être découpé une
fois de plus, et la trace montre alors un identifiant à trois étages. Cette récursion est ce qui
distingue un triage d'un simple choix de codec par fichier.

Le nombre d'appels n'est pas un coût de compression : c'est le prix du triage, et il est rapporté
séparément du taux. Un triage qui gagne en octets en payant des dizaines d'appels n'est pas
gratuit — il déplace le coût du côté de la décision.

## 8. La certification : recette compacte et rejeu sans LLM

La recette est la liste ordonnée des opérations, sérialisée **sans les octets** : elle ne contient
que les décisions. Elle est signée par HMAC-SHA256 — la clé est pédagogique, ce qui est signé est
l'**intégrité de la recette**, pas l'identité d'un émetteur.

Le rejeu parcourt la recette **à l'envers** : un `split` se reconstruit en recollant ses enfants
(les identifiants portent la structure), un `compress` se défait par le décodeur inverse. Aucun
appel au modèle n'intervient dans cette phase : c'est le contrat — *le LLM n'est pas nécessaire à
la décompression*.

In [9]:
static byte[] CLE_RECETTE = Encoding.UTF8.GetBytes("cle-pedagogique-compression-07");

static string Echapper(string s)
{
    var sb = new StringBuilder("\"");
    foreach (char c in s)
    {
        if (c == '"' || c == '\\') sb.Append('\\').Append(c);
        else if (c < 32) sb.Append("\\u").Append(((int)c).ToString("x4"));
        else sb.Append(c);
    }
    return sb.Append('"').ToString();
}

static byte[] RecetteCompacte(List<Dictionary<string, object>> ops)
{
    /// La recette sans les octets : les decisions, et rien d'autre — cles triees, forme canonique.
    var morceaux = ops.Select(op =>
    {
        if ((string)op["op"] == "compress")
            return $"{{\"codec\":{Echapper((string)op["codec"])},\"op\":\"compress\",\"seg\":{Echapper((string)op["seg"])}}}";
        var ids = (List<string>)op["enfants"];
        return $"{{\"enfants\":[{string.Join(",", ids.Select(Echapper))}],\"op\":\"split\"," +
               $"\"seg\":{Echapper((string)op["seg"])},\"strategie\":{Echapper((string)op["strategie"])}}}";
    });
    return Encoding.UTF8.GetBytes("[" + string.Join(",", morceaux) + "]");
}

static string Hmac(byte[] cle, byte[] corps) =>
    Convert.ToHexString(HMACSHA256.HashData(cle, corps)).ToLowerInvariant();

static Dictionary<string, object> Certifier(List<Dictionary<string, object>> ops, List<Segment> racines)
{
    var corps = RecetteCompacte(ops);
    var empreinte = SHA256.HashData(racines.SelectMany(s => s.Data).ToArray());
    return new Dictionary<string, object>
    {
        ["recette"] = corps,
        ["mac"] = Hmac(CLE_RECETTE, corps),
        ["empreinte_entree"] = Convert.ToHexString(empreinte).ToLowerInvariant(),
    };
}

Dictionary<string, byte[]> Archiver(Dictionary<string, Segment> index, List<Dictionary<string, object>> ops)
{
    /// Les blobs compresses, un par segment terminal.
    return ops.Where(op => (string)op["op"] == "compress")
        .ToDictionary(op => (string)op["seg"], op => CODECS[(string)op["codec"]](index[(string)op["seg"]].Data));
}

Dictionary<string, byte[]> Rejouer(List<Dictionary<string, object>> ops, Dictionary<string, byte[]> archive, List<Segment> racines)
{
    /// Decompression : la recette a l'envers, sans LLM.
    var data = new Dictionary<string, byte[]>();
    foreach (var op in Enumerable.Reverse(ops))
    {
        if ((string)op["op"] == "compress")
            data[(string)op["seg"]] = DECODEURS[(string)op["codec"]](archive[(string)op["seg"]]);
        else
        {
            var ids = (List<string>)op["enfants"];
            data[(string)op["seg"]] = ReconstruireEnfants(op, ids.Select(c => data[c]).ToList());
        }
    }
    return racines.ToDictionary(r => r.Id, r => data[r.Id]);
}

var ARCHIVE = Archiver(INDEX, OPS);
var CERTIFICAT = Certifier(OPS, racines);
var RECONSTRUIT = Rejouer(OPS, ARCHIVE, racines);
bool identiques = racines.All(r => RECONSTRUIT[r.Id].SequenceEqual(r.Data));
bool macValide = Hmac(CLE_RECETTE, (byte[])CERTIFICAT["recette"]) == (string)CERTIFICAT["mac"];
Console.WriteLine($"round-trip octet pour octet : {identiques}");
Console.WriteLine($"MAC de la recette valide    : {macValide}");
Console.WriteLine($"recette                     : {((byte[])CERTIFICAT["recette"]).Length} o signes");

round-trip octet pour octet : True


MAC de la recette valide    : True


recette                     : 914 o signes


### Lecture.

Deux vérifications indépendantes, et il faut les deux : le **round-trip** prouve que la recette
décrit fidèlement ce qui a été fait aux octets ; le **MAC** prouve que la recette rejouée est
celle qui a été signée, et non une recette modifiée après coup. Un round-trip réussi avec un MAC
faux signerait une recette altérée qui se trouve encore décompresser — exactement le cas qu'un
registre de recettes doit refuser.

L'égalité est exigée sur les **fichiers** reconstruits, pas sur les segments intermédiaires : un
enfant de `json_keys` est resérialisé lors du rejeu, et sa valeur suffit — le parent la relit, il
ne la compare pas octet par octet. Exiger davantage serait exigé à tort ; exiger moins laisserait
passer une recette qui perd du contenu. C'est la racine qui est le témoin, parce que c'est elle
que l'utilisateur avait avant de compresser.

## 9. Le banc comparatif

Trois totaux, sur le même contenu :

- **bloc, zlib** — le traitement indifférencié le plus courant ;
- **bloc, meilleur codec** — un oracle : on choisit après coup le meilleur codec global, ce que
  personne ne sait faire sans essayer tous les codecs sur tout le contenu ;
- **triage** — la somme des segments terminaux **plus la recette certifiée**, qui est le coût
  réel des décisions.

Le verdict est calculé dans la cellule, pas écrit à la main : un chiffre recopié dérive au premier
changement de corpus.

In [10]:
// Banc de mesure : compare le triage par segment a deux references traitees en bloc --
// zlib naif (tout concatener, compresser) et l'oracle du meilleur codec sur le bloc --
// fichier par fichier puis globalement, verdict inclus.
Dictionary<string, object> Banc(Dictionary<string, byte[]> fichiers, List<Dictionary<string, object>> ops,
    Dictionary<string, byte[]> archive, Dictionary<string, object> certificat, List<Segment> racines)
{
    var bloc = fichiers.Values.SelectMany(v => v).ToArray();
    var totaux = CODECS.ToDictionary(kv => kv.Key, kv => kv.Value(bloc).Length);
    var parRacine = new Dictionary<string, int>();
    foreach (var op in ops)
        if ((string)op["op"] == "compress")
        {
            string racine = ((string)op["seg"]).Split('.')[0];
            parRacine[racine] = parRacine.GetValueOrDefault(racine) + archive[(string)op["seg"]].Length;
        }
    var detail = new Dictionary<string, object>();
    foreach (var r in racines)
    {
        string nom = r.Kind.Split("fichier:")[^1];
        detail[nom] = new Dictionary<string, int>
        {
            ["entier_meilleur"] = CODECS.Values.Min(c => c(r.Data).Length),
            ["triage"] = parRacine.GetValueOrDefault(r.Id),
        };
    }
    return new Dictionary<string, object>
    {
        ["octets_entree"] = bloc.Length,
        ["bloc_zlib"] = totaux["zlib"],
        ["bloc_meilleur"] = totaux.Values.Min(),
        ["bloc_meilleur_codec"] = totaux.Aggregate((a, b) => a.Value <= b.Value ? a : b).Key,
        ["triage"] = archive.Values.Sum(b => b.Length) + ((byte[])certificat["recette"]).Length,
        ["recette"] = ((byte[])certificat["recette"]).Length,
        ["detail"] = detail,
    };
}

var RESULTAT = Banc(FICHIERS, OPS, ARCHIVE, CERTIFICAT, racines);
foreach (var cle in new[] { "octets_entree", "bloc_zlib", "bloc_meilleur", "bloc_meilleur_codec", "triage", "recette" })
    Console.WriteLine($"{cle,-22} {RESULTAT[cle]}");
Console.WriteLine("\npar fichier : meilleur codec sur le fichier entier contre triage par segment");
foreach (var (nom, d) in (Dictionary<string, object>)RESULTAT["detail"])
{
    var dd = (Dictionary<string, int>)d;
    int ecart = dd["entier_meilleur"] - dd["triage"];
    Console.WriteLine($"  {nom,-14} {dd["entier_meilleur"],7} -> {dd["triage"],7}  ({ecart:+0;-#})");
}
double gainNaif = (double)(int)RESULTAT["bloc_zlib"] / (int)RESULTAT["triage"];
double gainOracle = (double)(int)RESULTAT["bloc_meilleur"] / (int)RESULTAT["triage"];
Console.WriteLine($"\ntriage / bloc zlib      : {gainNaif.ToString("F2", CultureInfo.InvariantCulture)}x");
Console.WriteLine($"triage / bloc meilleur  : {gainOracle.ToString("F2", CultureInfo.InvariantCulture)}x");
string verdict = gainOracle > 1 ? "le triage gagne sur les deux traitements d'un bloc"
    : gainNaif > 1 ? "le triage gagne sur le traitement naif mais pas sur l'oracle global"
    : "le triage ne gagne sur aucun des deux traitements d'un bloc";
Console.WriteLine("verdict : " + verdict);

octets_entree          150748


bloc_zlib              20894


bloc_meilleur          12209


bloc_meilleur_codec    lzma


triage                 11013


recette                914


... (4 outputs left out)

  vrac.bin          6155 ->    6155  (+0)



triage / bloc zlib      : 1.90x


triage / bloc meilleur  : 1.11x


verdict : le triage gagne sur les deux traitements d'un bloc


### Lecture.

Le gain contre le bloc `zlib` est structurel : un seul codec ne peut pas être bon à la fois sur un
journal, sur des colonnes de natures différentes et sur un binaire déjà compressé. Le gain contre
l'**oracle** est le seul qui compte vraiment, parce que l'oracle a droit à tous les codecs sur
tout le contenu.

Le tableau par fichier dit d'où vient ce gain, et surtout **où il ne vient pas**. Une colonne de
catégories à cinq valeurs et une colonne d'entiers séquentiels se compriment presque à rien une
fois séparées ; entremêlées dans les lignes, elles forment un texte que même `lzma` ne ramène
pas. Le LLM n'a pas inventé ce gain — il a **choisi de découper**, ce qui est précisément le
triage qu'on lui demande.

Mais un découpage n'est pas gratuit, et le tableau le montre aussi : un fichier y apparaît avec un
écart **négatif**. Découper un petit objet en plusieurs clés, puis payer un blob et une entrée de
recette pour chacune, coûte parfois plus que le compresser d'un bloc. C'est le résultat honnête
du banc : le triage gagne en moyenne parce qu'il sépare ce qui doit l'être, et il perd localement
là où le modèle a découpé sans profit. Un triage parfait n'existe pas ; c'est la comparaison des
deux colonnes qui rend le sien vérifiable, fichier par fichier, plutôt qu'un chiffre global qu'on
pourrait améliorer en changeant de corpus.

Ce que le banc ne dit pas : que le LLM choisirait toujours bien. Un seul corpus, un seul modèle,
une seule graine. L'exercice 3 renverse la question et demande de mesurer un désaccord.

## À retenir

- Une **opération réelle** se définit par sa table inverse : `CODECS` et `DECODEURS` sont la même
  décision écrite deux fois, et le round-trip est ce qui le prouve.
- Une **opération intermédiaire** ne compresse pas : elle produit des segments qualifiés dont les
  identifiants portent la structure, ce qui permet de les recoller sans les décrire à nouveau.
- Le **LLM trie**, il ne compresse pas. Sa capacité sert à reconnaître la nature d'un segment et à
  décider s'il faut le découper ; les octets sont traités par les codecs.
- La **décompression est un rejeu**, pas une inférence : recette signée, exécution à l'envers,
  zéro appel au modèle.
- Le **coût des décisions** est dans le total. Une recette gratuite n'existe pas ; une recette
  qu'on oublie de compter fait gagner tous les bancs.

## Exercices

### Exercice 1 — Une primitive de plus : `train` (dictionnaire partagé)

`zlib` sait utiliser un dictionnaire (`zdict`). Ajouter une primitive `zlib_dict` qui prend, en
plus du segment, un dictionnaire de 256 octets, et l'enregistrer dans `CODECS` / `DECODEURS`.
Vérifier que le round-trip tient, puis mesurer si le dictionnaire aide sur les **petits** segments
produits par un découpage `lines` — c'est là qu'un dictionnaire partagé gagne sa place.

Côté .NET, `ZLibStream` n'expose pas `zdict` : l'exercice demande d'identifier la voie qui le
permet (appel bas niveau, bibliothèque tierce) et de la mesurer — la limite de la BCL fait partie
de l'exercice.

In [11]:
static byte[] DecoderZlibDict(byte[] blob, byte[] zdict)
{
    /// Exercice 1 : a completer -- decompression zlib avec dictionnaire.
    return null;  // TODO etudiant : completer, puis verifier le round-trip
}

### Exercice 2 — Une stratégie de plus : `records` (transposer un JSON de lignes)

Un JSON qui contient une **liste d'objets de même forme** (`mesures`) se comprime mieux transposé :
une clé devient une colonne. Écrire une stratégie `records` qui produit un segment par champ,
comme `csv_columns`, puis écrire son inverse. Mesurer sur `mesures.json` du banc : la transposition
bat-elle le `lzma` que le LLM a choisi ?

In [12]:
static List<Segment> DecouperRecords(Segment seg)
{
    /// Exercice 2 : a completer -- transposer une liste d'objets en colonnes.
    return new List<Segment>();  // TODO etudiant : completer
}

### Exercice 3 — Mesurer un désaccord

Le banc ne montre qu'un corpus. Construire un second corpus où le triage **perd** contre l'oracle
global — par exemple un répertoire de fichiers homogènes de même nature, où le découpage ne fait
que payer ses en-têtes. Rapporter le chiffre, la cause, et ce que cela dit de la limite du triage :
un triage n'est pas une compression, c'est une **allocation** de codecs et de frontières.

In [13]:
static Dictionary<string, byte[]> CorpusHomogene()
{
    /// Exercice 3 : a completer -- un corpus ou le triage doit perdre.
    return new Dictionary<string, byte[]>();  // TODO etudiant : completer, puis comparer au banc
}

## Ce que le carnet laisse ouvert

- Le triage est **séquentiel** : chaque segment est décidé seul. Un modèle qui verrait l'ensemble
  des segments d'un coup pourrait choisir des frontières cohérentes — c'est le sujet d'un carnet
  suivant, pas de celui-ci.
- Le choix du codec par le LLM n'est pas vérifié optimal : la mesure le sanctionne, elle ne
  l'explique pas.
- La recette est signée pour son intégrité, pas pour son authenticité : un émetteur qui partage
  la clé pédagogique peut signer n'importe quoi. Un registre réel sépare émetteur et vérificateur.

## Vérifiez votre compréhension

- La certification exige **à la fois** le round-trip et le MAC. Que prouve
  chacun séparément, et que manquerait-il à la preuve si on ne gardait que
  l'un des deux ?
- Le LLM ne « compresse » rien : il **décide**. En quoi la recette compacte —
  les décisions, sans les octets — est-elle précisément ce qui rend le rejeu
  possible **sans modèle** ?
- Le banc oppose triage / bloc-zlib / oracle. Pourquoi le gain **contre
  l'oracle** est-il l'information la plus honnête à lire, et que dit-il sur ce
  que le triage « sait » (ou ne sait pas) du corpus ?

## Pour aller plus loin

- [Compression-06 — MDL et complexité de Kolmogorov](Compression-06-MDL-Kolmogorov-CSharp.ipynb) :
  le triage est une heuristique de MDL ; relier la décision du LLM au critère
  de longueur de description qu'il approxime.
- Le triage est **séquentiel** (chaque segment décidé seul, cf. le bloc « ce que
  le carnet laisse ouvert ») : qu'un modèle qui verrait l'ensemble des segments
  d'un coup gagnerait-il en cohérence de frontières ?
- Les stratégies de découpage (`lines`, `blocks`, `csv_columns`, `json_keys`) :
  en écrire une pour un format que le banc ne couvre pas (paires `clé=valeur`,
  `parquet`, …) et mesurer ce que le triage en fait.